### Récupération des données et nettoyage


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle as pck
import plotly.express as px
import ydata_profiling as ydp
from ydata_profiling import ProfileReport

#Création des Dataframes pour chaque csv
aisles_df = pd.DataFrame(pd.read_csv('../INPUT_DATA/aisles.csv'))
departments_df = pd.DataFrame(pd.read_csv('../INPUT_DATA/departments.csv'))
orders_df = pd.DataFrame(pd.read_csv('../INPUT_DATA/orders.csv'))
products_df = pd.DataFrame(pd.read_csv('../INPUT_DATA/products.csv'))
orderproducts_df = pd.DataFrame(pd.read_csv('../INPUT_DATA/order_products.csv'))


#Conversion par précaution des order_dow et hour en int
orders_df['order_hour_of_day'] = orders_df['order_hour_of_day'].astype(int)
orders_df['order_dow'] = orders_df['order_dow'].astype(int)
orders_df.fillna(0, inplace=True)

# orders_df
#Création de la liaison entre les produits et leurs départements et allées
products_full = products_df.merge(aisles_df, on="aisle_id").merge(departments_df, on="department_id")

#Création de la liaison entre les produits et les commandes
orderproducts_full = orderproducts_df.merge(products_full, on="product_id")


#Création d'une dataframe avec toutes les données et dans les id de products, aisle, department
orders_products_fusion = orderproducts_full.copy()


#Liaison entre orders products nettoyé et orders
orders_products_final = orders_df.merge(orders_products_fusion, on="order_id")
# orders_products_final

# orderproducts_df



### Mappage des jours de commandes


In [ ]:
orders_products_map = orders_products_final.copy()


order_day_mapping = {
    0: "Mond",
    1: "Tuesd",
    2: "Wednesd",
    3: "Thursd",
    4: "Frid",
    5: "Saturd",
    6: "Sund"
}

orders_products_map["order_days"] = orders_products_map["order_dow"].map(order_day_mapping)

orders_products_map.drop(columns="order_dow", inplace=True)




# orders_products_map
# data_profile = ProfileReport(orders_products_map, title="My Data Profiling Report", html={'style': {'full_width':True}})

# data_profile.to_notebook_iframe()

# data_profile.to_file("report.html")
# orders_products_map.to_parquet('orders_products_map.parquet', engine="fastparquet", index=False)


### Structuration des données


In [ ]:
structure_of_data = {
    'Type': ['Customers', 'Orders', 'Products', 'Aisles', 'Departments'],
    'Count': [
        len(orders_products_map.user_id.unique()),
        len(orders_products_map.order_id.unique()),
        len(orders_products_map.product_id.unique()),
        len(orders_products_map.aisle_id.unique()),
        len(orders_products_map.department_id.unique())
    ]
}

data_structure = pd.DataFrame(structure_of_data)
# data_structure.to_csv("./OUTPUT/data_structure.csv")
# data_structure

fig = px.bar(data_structure, x='Type', y='Count', text='Count', title='Structure des données')
fig.update_traces(textposition='outside')
fig.show()



Données :
	•	206 209 clients ont passé au total 3,2 millions de commandes.
	•	Le catalogue du site contient près de 49 677 produits, répartis dans 134 allées et 21 départements.

Ce qui veut dire qu’on est face à un site très complet, avec une base d’utilisateurs importante et fidèle.
Le volume de commandes est largement supérieur au nombre de clients, ce qui prouve qu’il y a un vrai effet de réachat.
Autrement dit, les clients reviennent régulièrement faire leurs courses sur le site.

### Nombre de commandes par jour de la semaine


In [ ]:
# Commandes par jour
orders_by_day = orders_products_map.groupby("order_days")["order_id"].nunique()
orders_by_day = orders_by_day.reset_index()
orders_by_day = orders_by_day.rename(columns={"order_id": "count_orders"})
# orders_by_day.to_csv("./OUTPUT/orders_by_day.csv")

order_fig = ["Mond","Tuesd","Wednesd","Thursd","Frid","Saturd","Sund"]


orders_by_day_fig = px.bar(orders_by_day, x='order_days', y='count_orders', color='count_orders', height=500, width=800, title="Nombre de commandes par jour de la semaine", category_orders={"order_days": order_fig})
orders_by_day_fig.update_layout(
    xaxis_title="DAYS",
    yaxis_title="NUMBER OF ORDERS",
)
orders_by_day_fig.show()




Le pic d’activité se situe le lundi (557 772 commandes) et le mardi (556 705).
Les jours les plus calmes sont le vendredi (401 212) et le jeudi (412 400).

On voit que les utilisateurs commandent surtout en début de semaine, probablement pour planifier leurs repas et remplir leur frigo après le week-end.
Cela montre une habitude d’achat assez stable, proche du comportement qu’on retrouve dans les supermarchés physiques : les clients font leur “plein” en début de semaine.

À conseiller :

Les lundis et mardis sont les jours stratégiques pour lancer des promotions sur les produits du quotidien ou envoyer des emails de relance.

### Nombre de commandes par heures de la journée


In [ ]:
# Commandes par heure
orders_by_hour = orders_products_map.groupby("order_hour_of_day")["order_id"].nunique()
orders_by_hour = orders_by_hour.reset_index()
orders_by_hour = orders_by_hour.rename(columns={"order_id": "count_orders"})
# orders_by_hour.to_csv("./OUTPUT/orders_by_hour.csv")


orders_by_hour_fig = px.bar(orders_by_hour, x='order_hour_of_day', y='count_orders', color='count_orders', height=500, width=800, title="Nombre de commandes par heure de la journée")
orders_by_hour_fig.update_layout(
    xaxis_title="HOURS",
    yaxis_title="NUMBER OF ORDERS",
)
orders_by_hour_fig.show()



Les heures les plus actives sont 10h à 15h, avec un pic entre 10h et 12h (près de 270 000 commandes par heure).
Les heures les plus calmes sont la nuit (2h à 5h).

Globalement, les clients commandent en journée, souvent sur leur pause ou juste avant le déjeuner.
Cela confirme que le site est principalement utilisé sur ordinateur ou mobile pendant les heures de travail.

À conseiller :

Les campagnes de communication ou les notifications doivent être programmées autour de 10h ou 11h, quand les gens sont connectés et disponibles.

### Répartition des commandes d'une journée en fonction des heures


In [ ]:
orders_by_day_and_hours = orders_products_map.groupby(["order_days", "order_hour_of_day"])["order_id"].nunique()
orders_by_day_and_hours = orders_by_day_and_hours.reset_index()
orders_by_day_and_hours = orders_by_day_and_hours.rename(columns={"order_id": "count_orders"})
# orders_by_day_and_hours.to_csv("./OUTPUT/orders_by_day_and_hours.csv")


orders_by_day_and_hours_fig = px.line(orders_by_day_and_hours, x='order_hour_of_day', y='count_orders', color='order_days', height=500, width=800, title="Nombre de commandes par heure pour chaque jour")
orders_by_day_and_hours_fig.update_layout(
    xaxis_title="Hours (0–23)",
    yaxis_title="NUMBER OF ORDERS",
)
orders_by_day_and_hours_fig.show()



La distribution jour/heure confirme que le cœur de l’activité se situe entre 9h et 16h, principalement les lundis, mardis et mercredis.
Les week-ends ont un volume un peu plus faible, ce qui peut indiquer que les clients préfèrent anticiper plutôt que de faire leurs courses à la dernière minute.

### Quel département vend le plus ?


In [ ]:
dept_sales = orders_products_map.groupby("department")["product_id"].count().sort_values(ascending=False)
dept_sales = dept_sales.reset_index()
dept_sales = dept_sales.rename(columns={"product_id": "count_product"})

# dept_sales.head(20).to_csv("./OUTPUT/dept_sales.csv")


dept_sales_fig = px.bar(dept_sales.head(20), x='count_product', y='department', color='count_product', height=500, width=800, title="Départements faisant le plus de vente (TOP 20)")
dept_sales_fig.update_layout(
    xaxis_title="DEPARTMENT",
    yaxis_title="NUMBER OF PRODUCTS SOLD",
)
dept_sales_fig.show()

Le département “produce”est de loin le plus important, avec 9,4 millions d’articles vendus.
Il est suivi par :
	•	Dairy & eggs (5,4 M)
	•	Snacks (2,8 M)
	•	Beverages (2,6 M)
	•	Frozen (2,2 M)

On voit clairement que les achats sont centrés sur l’alimentaire quotidien : fruits, légumes, lait, œufs, boissons, produits congelés.
Les autres départements (hygiène, maison, soins) restent secondaires.

À conseiller :

Les produits frais sont le cœur du business.
Ce sont eux qui amènent les clients à revenir — il faut donc maintenir une excellente qualité de service et de stock sur ces catégories.

### Quel allée vend le plus ?


In [ ]:
aisle_sales = orders_products_map.groupby("aisle")["product_id"].count().sort_values(ascending=False)
aisle_sales = aisle_sales.reset_index()
aisle_sales = aisle_sales.rename(columns={"product_id": "count_product"})

aisle_sales_fig = px.bar(aisle_sales.head(20), x='count_product', y='aisle', color='count_product', height=500, width=800, title="Allées faisant le plus de vente (TOP 20)")
aisle_sales_fig.update_layout(
    xaxis_title="AISLE",
    yaxis_title="NUMBER OF PRODUCTS SOLD",
)
aisle_sales_fig.show()

Les allées les plus actives sont :
	•	Fresh fruits (3,64 M ventes)
	•	Fresh vegetables (3,41 M)
	•	Packaged vegetables & fruits (1,76 M)
	•	Yogurt (1,45 M)
	•	Packaged cheese, milk, and bread suivent de près.

Ces chiffres montrent que les achats tournent autour des produits frais et sains.
Les allées “chips”, “soft drinks” ou “snacks” sont bien présentes mais restent secondaires.

À conseiller :

Les clients sont sensibles à la fraîcheur et à la qualité des produits.
Le site doit continuer à valoriser les produits frais et les gammes bio, qui attirent une clientèle fidèle.

### Quels sont les produits les plus vendus ?


In [ ]:

prod_sales_to = orders_products_map["product_name"].value_counts().head(30)
prod_sales_to = prod_sales_to.reset_index()
prod_sales_to = prod_sales_to.rename(columns={"count": "count_product"})
prod_sales = prod_sales_to.iloc[::-1]

prod_sales_fig = px.bar(prod_sales, x='count_product', y='product_name', color='count_product', height=800, width=1500, title="Produits faisant le plus de vente (TOP 30)")
prod_sales_fig.update_layout(
    xaxis_title="PRODUCTS",
    yaxis_title="NUMBER OF PRODUCTS SOLD",
)
prod_sales_fig.show()



Le produit le plus vendu est sans surprise la banane (472 565 unités), suivie du sachet de bananes bio, des fraises bio et de l’avocat bio.

On remarque que le Top 10 est presque entièrement composé de produits biologiques :
“Organic Baby Spinach”, “Organic Avocado”, “Organic Strawberries”, “Organic Blueberries”…

Cela confirme encore une fois que la clientèle du site a une forte affinité avec les produits sains, naturels et bio.

À consiller :

Le site attire une clientèle à la recherche de qualité, fraîcheur et naturalité.
Il faut donc maintenir une offre bio riche et visible.

### Quels sont les produits les moins vendus ?


In [ ]:

prod_sales_le = orders_products_map["product_name"].value_counts().tail(30)
prod_sales_le = prod_sales_le.reset_index()
prod_sales_le = prod_sales_le.rename(columns={"count": "count_product"})

# prod_sales.head(30).to_csv("./OUTPUT/prod_least_sales.csv")

prod_sales_le_fig = px.bar(prod_sales_le, x='count_product', y='product_name', color='count_product', height=800, width=1500, title="Produits faisant le moins de vente (TOP 30)")
prod_sales_le_fig.update_layout(
    xaxis_title="PRODUCTS",
    yaxis_title="NUMBER OF PRODUCTS SOLD",
)
prod_sales_le_fig.show()



Contrairement aux produits précédents, certains produits n’ont été achetés qu’une seule fois (ex : “Greek Yogurt Cherry”, “Kefir Raspberry”, “Coconut Butter”…).
Il s’agit souvent de produits très spécifiques ou de niches peu connues.

Ce faible volume n’est pas toujours un problème : certains produits de niche peuvent servir d’appel pour attirer une clientèle précise.
Mais d’autres peuvent simplement être mal référencés ou mal positionnés sur le site.

À conseiller :

Une révision du catalogue et du référencement des produits serait utile pour améliorer la visibilité des articles à très faible taux d'achats.

### Quels sont les produits les plus vendus par département ?


In [ ]:

dep_prod = (orders_products_map.groupby(["department","product_name"]).size().reset_index(name="sales"))
top_by_dep = (dep_prod.sort_values(["department","sales"], ascending=[True, False]).groupby("department").head(1))
top_by_dep.sort_values(["sales"], ascending=[True], inplace=True)

top_by_dep['department-product'] = top_by_dep['department'] + '-//-' + top_by_dep['product_name']

# top_by_dep.to_csv("./OUTPUT/top_by_dep.csv")

top_by_dep_fig = px.pie(top_by_dep, values='sales', names='department-product', title='Camembert Des produits les Plus Vendus par départements', width=1500, height=1000)
top_by_dep_fig.show()

# top_by_dep



### Quels sont les produits les plus vendus par allée ?


In [ ]:
aisle_prod = (orders_products_map.groupby(["aisle","product_name"]).size().reset_index(name="sales"))
top_by_aisle = (aisle_prod.sort_values(["aisle","sales"], ascending=[True, False]).groupby("aisle").head(1))
top_by_aisle.sort_values(["sales"], ascending=[True], inplace=True)


top_by_aisle['aisle-product'] = top_by_aisle['aisle'] + '-//-' + top_by_aisle['product_name']

# top_by_aisle.to_csv("./OUTPUT/top_by_aisle.csv", index=False)

top_by_aisle_fig = px.pie(top_by_aisle.head(20), values='sales', names='aisle-product', title='Camembert Des produits les Plus Vendus par allée', width=1500, height=1000)
top_by_aisle_fig.show()

# top_by_aisle



Dans chaque département, on retrouve un produit phare :
	•	“Banana” dans produce,
	•	“Organic Whole Milk” dans dairy & eggs,
	•	“Sparkling Water Grapefruit” dans beverages.

Ces produits constituent l’ADN du site.
Ce sont eux qui symbolisent ce que le client vient chercher : du frais, du bio et des produits essentiels.

À conseiller :

Ces produits doivent rester en tête de rayon, visibles sur la page d’accueil et en recommandation automatique.

### Premiers produits ajoutés au panier (que recherche le client en premier)


In [ ]:

first_product = orders_products_map.loc[orders_products_map["add_to_cart_order"]==1, "product_name"].value_counts()
first_product = first_product.reset_index()

first_product_fig = px.bar(first_product.head(20).sort_values('count', ascending=True), x="count", y="product_name", height=500, orientation="h", title="Produits ajoutés au panier en premier (Top 20)", labels={"count": "Occurrences", "product_name": "Produit"}, color="count")
first_product_fig.update_layout(
    xaxis_title="NUMBER OF PRODUCTS FIRST ADDED TO CART",
    yaxis_title="PRODUCTS NAME",
)
first_product_fig.show()


Les clients ajoutent les produits du quotidien en premier dans leur panier :
	•	Banane,
	•	Lait entier bio,
	•	Fraises,
	•	Avocat,
	•	Épinards bio.

Cela montre que les utilisateurs viennent avec une intention d’achat claire : faire leurs courses de base.
Ces produits sont des “déclencheurs d’achat”.

À conseiller :

Ces articles doivent être facilement accessibles dès la page d’accueil ou dans un widget “vos achats habituels” ou dans une popup en CTA avec des promotions.

### Derniers produits ajoutés au panier (que recherche le client en dernier)


In [ ]:

max_add = orders_products_map.groupby("order_id")["add_to_cart_order"].transform("max")

last_product = orders_products_map.loc[orders_products_map["add_to_cart_order"]==max_add, "product_name"].value_counts().head(20).reset_index()
# last_product.head(20).sort_values().to_csv("./OUTPUT/last_product.csv")


last_product_fig = px.bar(last_product.sort_values("count"), x="count", y="product_name", height=500, orientation="h", title="Produits ajoutés en dernier au panier (Top 20)", labels={"count": "Occurrences", "product_name": "Produit"}, color="count")
last_product_fig.update_layout(
    xaxis_title="NUMBER OF PRODUCTS LATEST ADDED TO CART",
    yaxis_title="PRODUCTS NAME",
)
last_product_fig.show()



### Meilleurs produits par jour


In [ ]:
# Meilleurs produits par jour
prod_by_day = (orders_products_map.groupby(["order_days","product_name"]).size().reset_index(name="sales"))
top_prod_by_day = (prod_by_day.sort_values(["order_days","sales"], ascending=[True, False]).groupby("order_days").head(15))

# top_prod_by_days.to_csv("./OUTPUT/top_prod_by_days.csv")

fig = px.bar(top_prod_by_day, x="order_days", y="sales", color="product_name", title="Top 15 produits les plus vendus par jour de la semaine", labels={"order_days": "Jour de la semaine", "sales": "Nombre de ventes", "product_name": "Produit"}, barmode="group")

fig.update_layout(xaxis=dict(categoryorder="array", categoryarray=["Mond","Tuesd","Wednesd","Thursd","Frid","Saturd","Sund"]), legend_title_text="Produit")

fig.show()


Les produits ajoutés en dernier sont souvent les mêmes, mais en quantités plus faibles, cela est très certainement dû aux grandes quantités de ces produits qui sont achetées :
	•	Banane,
	•	Fraises,
	•	Légumes bio,
	•	Avocat,
Cependant, des exceptions se retrouvent dans cette catégorie comme :
	•	Eau ou soda.

Cela correspond à des achats de complément : des produits auxquels on pense à la fin.

À conseiller :

On pourrait placer des suggestions intelligentes à la fin du panier (“Il vous manque peut-être…”), des sortes de CTA.

### Bio VS NON-BIO


In [ ]:
is_org = orders_products_map["product_name"].str.contains("organic", case=False, na=False)
org_counts = pd.Series({"organic": int(is_org.sum()), "not_organic": int((~is_org).sum())}).reset_index()
org_counts.columns = ["Type", "Count"]
org_counts.columns = ["Type", "Count"]

# org_counts.to_csv("./OUTPUT/organic_counts.csv")

org_counts_fig = px.pie(org_counts, names="Type", values="Count", title="Bio vs Non-Bio", color="Type", color_discrete_map={"Bio": "green", "Non Bio": "gray"})

org_counts_fig.update_traces(textinfo="percent+label")

org_counts_fig.show()


Les produits bio représentent 10,25 millions d’achats, contre 22,18 millions pour le non-bio.
Cela veut dire qu’environ 32 % des ventes concernent des produits bio.

C’est un taux très élevé : près d’un achat sur trois est bio.
Le site attire donc une clientèle attentive à l’origine et la qualité des produits.

À conseiller :

Le “bio” est une force commerciale majeure du site.
Il faut le mettre en avant dans le parcours client, et enrichir cette gamme en magasin.

### Taux de Rachat


In [ ]:

prod_reorder = orders_products_map.groupby("product_name")["reordered"].agg(['mean','count'])
prod_reorder = prod_reorder.rename(columns={'mean':'reorder_rate','count':'n'})


stable = prod_reorder[prod_reorder["n"] >= 10]
top_reorder = stable.sort_values("reorder_rate", ascending=False).head(30).reset_index()


# top_reo.to_csv("./OUTPUT/top_products_reorder.csv")

top_reorder_fig = px.bar(top_reorder.sort_values("reorder_rate"), x="reorder_rate", y="product_name", orientation="h", title="Top 30 produits avec le plus fort taux de rachat (n ≥ 10)", labels={"reorder_rate": "Taux de rachat", "product_name": "Produit"}, color="reorder_rate", color_continuous_scale="Viridis")

top_reorder_fig.update_layout(xaxis_tickformat=".0%", yaxis_title="", xaxis_title="Probabilité de rachat (reorder rate)")

top_reorder_fig.show()




Les meilleurs taux de rachat dépassent les 90 %, comme :
	•	“Raw Veggie Wrappers” (0.94)
	•	“Chocolate Love Bar” (0.92)
	•	“Soy Powder Infant Formula” (0.91)
	•	“Organic Blueberry B Mega” (0.88)

Ces taux montrent que certains produits fidélisent énormément les clients.
Ce sont souvent des produits de qualité, consommés régulièrement.

À conseiller :

Ces produits sont parfaits pour mettre en place des abonnements (“recevez ce produit chaque semaine/mois”)
ou des suggestions automatiques dans les paniers récurrents.

### A quelle fréquence les utilisateurs reviennent faire des achats ?


In [ ]:

user_freq_reorder = orders_products_map.groupby("user_id")["days_since_prior_order"].mean().reset_index()
freq_reorder_mean = user_freq_reorder["days_since_prior_order"].mean()
user_freq_reorder


# fig = px.histogram(user_freq_reorder, x="days_since_prior_order", nbins=30, title="Distribution de la fréquence moyenne de rachat par utilisateur", labels={"days_since_prior_order": "Jours moyens entre deux commandes", "count": "Nombre d'utilisateurs"})
#
# fig.update_layout( xaxis_title="Nombre moyen de jours entre deux commandes", yaxis_title="Nombre d'utilisateurs")
#
# fig.show()


### Conclusion :

L’analyse des données montre que :
	•	Les commandes sont concentrées en début de semaine et en journée, ce qui reflète un comportement d’achat bien ancré.
	•	Les produits frais, bio et du quotidien constituent le cœur de l’activité.
	•	Les clients reviennent régulièrement (environ une fois par semaine), preuve d’une forte fidélité.
	•	Certains produits et rayons dominent largement les ventes, ce qui indique un catalogue bien segmenté mais qui peut être amélioré.

En résumé :

Le site attire une clientèle fidèle, sensible à la qualité et au bio, qui fait ses courses de manière planifiée.
Pour aller plus loin, il faut renforcer la personnalisation, automatiser les suggestions de rachat, et optimiser la mise en avant des produits clés.

Au regard de ces résultats, les tendances à venir semblent s’orienter vers une augmentation des achats de produits bio et frais. Les consommateurs montrent un comportement de plus en plus prévisible et structuré, privilégiant la commodité et la rapidité. On peut donc s’attendre à une hausse des rachats automatiques (produits du quotidien, lait, fruits, légumes, eau), à une fidélisation accrue autour des produits bio et locaux, et à une demande croissante de personnalisation dans les recommandations.

Le site a donc tout intérêt à renforcer la personnalisation de l’expérience, à développer des rappels de commande intelligents, et à proposer des programmes de fidélité ciblés autour des produits les plus achetés. Ces initiatives permettront non seulement d’augmenter la fréquence d’achat, mais aussi de consolider la relation de confiance entre le site et ses clients.